In [8]:
# !pip install --upgrade pip
!pip install -r requirement.txt

  Using cached numpy-2.4.2-cp311-cp311-macosx_14_0_arm64.whl.metadata (6.6 kB)
  Using cached matplotlib-3.10.8-cp311-cp311-macosx_11_0_arm64.whl.metadata (52 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached contourpy-1.3.3-cp311-cp311-macosx_11_0_arm64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.61.1-cp311-cp311-macosx_10_9_universal2.whl.metadata (114 kB)
  Using cached kiwisolver-1.4.9-cp311-cp311-macosx_11_0_arm64.whl.metadata (6.3 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 5.2 MB/s  0:00:01m0:00:0100:01
Using cached numpy-2.4.2-cp311-cp311-macosx_14_0_arm64.whl (5.5 MB)
Using cached matplotlib-3.10.8-cp311-cp311-macosx_11_0_arm64.whl (8.1 MB)
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
Using cached contourpy-1.3.3-cp311-cp311-macosx_11_0_arm64.whl (270 kB)
Using cached cycler-0.12.1-py3-n

Load Data

In [134]:
import pandas as pd
import numpy as np

# Load data
df = pd.read_csv("data/company_metrics_data.csv")

# Convert month to datetime
df["month_key"] = pd.to_datetime(df["month_key"])

# Sort for time integrity
df = df.sort_values(["company_id", "month_key"])

print("Shape:", df.shape)
df.head()

Shape: (500, 16)


,company_id,month_key,avg_dpp_per_invoice,monthly_spend_volatility,total_dpp,total_inbound_tax_invoices,total_vat,ammend_rate,approval_rate,friction_value_ratio,hard_failure_rate,top_1_vendor_share,top_5_vendor_share,unique_vendors,vendor_concentration_index,vendor_repeat_rate
9,C001,2025-01-01,76306886,0.619,467014547,202,45262516,0.123,0.833,0.043,0.098,0.319,0.348,24,2822,0.893
5,C001,2025-02-01,76329229,0.471,56184743,213,5204012,0.010,0.564,0.426,0.053,0.127,0.826,100,1430,0.626
1,C001,2025-03-01,19642295,0.301,169521763,207,16600946,0.010,0.791,0.199,0.033,0.458,0.760,79,1910,0.582
8,C001,2025-04-01,89853482,0.028,585934177,188,55027863,0.160,0.786,0.055,0.023,0.251,0.439,21,1391,0.652
0,C001,2025-05-01,41683335,0.067,851630840,169,78054326,0.096,0.847,0.057,0.038,0.674,0.726,55,4134,0.493


In [141]:
import pandas as pd
import numpy as np
import json

df = pd.read_csv('data/company_metrics_data.csv')

df['month_key'] = pd.to_datetime(df['month_key'])

metrics = [
    'monthly_spend_volatility',
    'ammend_rate',
    'friction_value_ratio',
    'hard_failure_rate',
    'vendor_concentration_index'
]

df = df.sort_values(['company_id', 'month_key'])
zscore_df = []

for month in df['month_key'].sort_values().unique():
    historical = df[df['month_key'] <= month]
    temp = df[df['month_key'] == month].copy()
    
    for m in metrics:
        mean = historical[m].mean()
        std = historical[m].std(ddof=0)
        # Avoid division by zero
        if std == 0:
            temp[f'{m}_z'] = 0
        else:
            temp[f'{m}_z'] = (temp[m] - mean) / std
    zscore_df.append(temp)

df_z = pd.concat(zscore_df)

z_cols = [f'{m}_z' for m in metrics]
df_z['company_risk_score'] = df_z[z_cols].mean(axis=1)

def assign_profile_by_quantiles(group):
    q1 = group['company_risk_score'].quantile(0.25)
    q3 = group['company_risk_score'].quantile(0.75)
    
    def profile(score):
        if score < q1:
            return 'Low Risk'
        elif score > q3:
            return 'High Risk'
        else:
            return 'Moderate'
    
    group['risk_profile'] = group['company_risk_score'].apply(profile)
    return group

df_z = df_z.groupby('month_key').apply(assign_profile_by_quantiles).reset_index()

quantile_threshold = 0.75

month_thresholds = df_z.groupby('month_key')[
    ['hard_failure_rate_z', 'monthly_spend_volatility_z', 
     'ammend_rate_z', 'friction_value_ratio_z', 
     'vendor_concentration_index_z']
].quantile(quantile_threshold).to_dict()

def generate_alerts(row):
    alerts = []

    month = row['month_key']
    if row['hard_failure_rate_z'] > month_thresholds['hard_failure_rate_z'][month]:
        alerts.append("Low invoice quality")
    if row['monthly_spend_volatility_z'] > month_thresholds['monthly_spend_volatility_z'][month]:
        alerts.append("High spend volatility")
    if row['ammend_rate_z'] > month_thresholds['ammend_rate_z'][month]:
        alerts.append("High amendment rate")
    if row['friction_value_ratio_z'] > month_thresholds['friction_value_ratio_z'][month]:
        alerts.append("High friction")
    if row['vendor_concentration_index_z'] > month_thresholds['vendor_concentration_index_z'][month]:
        alerts.append("Vendor concentration risk")
    if len(alerts) == 0:
        alerts.append("No alerts")

    return alerts

df_z['alert'] = df_z.apply(generate_alerts, axis=1)

df_z['month_key'] = pd.to_datetime(df_z['month_key'])
df_z['month_key'] = df_z['month_key'].dt.strftime('%Y-%m-%d')

output_columns = ['company_id','month_key','company_risk_score','risk_profile','alert']
json_output = df_z[output_columns].to_dict(orient='records')

print(json.dumps(json_output, indent=2))

[
  {
    "company_id": "C001",
    "month_key": "2025-01-01",
    "company_risk_score": 0.8119545722384398,
    "risk_profile": "High Risk",
    "alert": [
      "Low invoice quality",
      "High spend volatility",
      "High amendment rate"
    ]
  },
  {
    "company_id": "C002",
    "month_key": "2025-01-01",
    "company_risk_score": 0.8465475945865369,
    "risk_profile": "High Risk",
    "alert": [
      "High spend volatility",
      "High amendment rate",
      "Vendor concentration risk"
    ]
  },
  {
    "company_id": "C003",
    "month_key": "2025-01-01",
    "company_risk_score": 0.5402820698554358,
    "risk_profile": "High Risk",
    "alert": [
      "High spend volatility",
      "Vendor concentration risk"
    ]
  },
  {
    "company_id": "C005",
    "month_key": "2025-01-01",
    "company_risk_score": -0.3522136300111508,
    "risk_profile": "Low Risk",
    "alert": [
      "No alerts"
    ]
  },
  {
    "company_id": "C006",
    "month_key": "2025-01-01",
    "com